# FFT from live CODEC audio

Companion to `fft_dma_demo.ipynb`. Instead of feeding samples into the
`fft_wrapper` HLS core from DDR over the DMA `MM2S` channel, this takes **live
audio from the SSM2603 codec** and runs it through the same FFT. A mux IP
(`dma_codec_mux_wrapper_0`) selects which stream drives the FFT input.

> **Run this as root.** This board uses a custom non-XRT PYNQ backend
> (`zynq_cma_device`) that does MMIO and DMA over `/dev/mem`, which needs root.
> The bitstream is already loaded at boot (it lives in `boot.bin`), so the PL is
> live and we only *attach* to it — we never reprogram it.

**Data path** (from `cfg/fft_demo_integ/bd_project.tcl`):

```
fft_dma.M_AXIS_MM2S ------------------\
                                       >-- dma_codec_mux --> fft_wrapper.input_signal_stream
sampler.axis_interface_master (CODEC) /   (sel 0x10: 0=DMA, 1=CODEC)

fft_wrapper.fft_output_stream --S2MM--> DDR
```

Because the codec is a *continuous* source, we only use the DMA **receive**
(`S2MM`) channel to capture FFT frames — there is no `MM2S` send. `fft_wrapper`
is armed with auto-restart so it keeps transforming frame after frame.

**Fixed-point formats** (from `cpp/fft_sysdef.h`), same as the DMA demo:

| signal | C++ type | bits | scale (1.0 =) |
|--------|----------|------|----------------|
| FFT output re/im | `ap_fixed<32,9>` | 32 each | `2**23` |
| window coeff   | `ap_ufixed<18,0>` | 18 | `2**18` |

Each `S2MM` beat is 64 bits: **lower int32 = real, upper int32 = imag**.

In [ ]:
# zynq_cma_device is a non-XRT shim shipped by meta-hls-pynq: importing it
# registers a /dev/mem-based Device as the active PYNQ device so Overlay/MMIO/
# allocate work. It MUST be imported before pynq, and needs root for /dev/mem.
import zynq_cma_device   # registers itself as the active PYNQ device

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

# Make the ported codec driver importable (lives next to the original C source).
for _cand in (
    os.path.join(os.getcwd(), '..', 'subsystems', 'codec_unit', 'fw'),
    os.getcwd(),   # if codec_controller.py was staged next to this notebook
):
    _cand = os.path.abspath(_cand)
    if os.path.exists(os.path.join(_cand, 'codec_controller.py')):
        sys.path.insert(0, _cand)
        break
from codec_controller import CodecController

# ---- design constants ----
N        = 256          # FFT points
FS       = 44100        # codec sample rate [Hz] -- matches codec init (USB mode,
                        # SAMPLE reg 0x23 -> ~44.1 kHz). Keep in sync with init().
OUT_FRAC = 23           # ap_fixed<32,9>  -> 32-9 fractional bits
WIN_FRAC = 18           # ap_ufixed<18,0> -> 18 fractional bits

BITSTREAM = 'fft_demo_top_wrapper.bit'
OUT_SCALE = float(1 << OUT_FRAC)

# ---- register offsets ----
AP_CTRL_OFFSET     = 0x00
WIN_COEFF_OFFSET   = 0x200   # <-- VERIFY against xfft_wrapper_hw.h
DMA_MUX_SEL_OFFSET = 0x10
MUX_SEL_DMA        = 0x0
MUX_SEL_CODEC      = 0x1

## 1. Attach to the already-loaded overlay

`download=False` parses the `.hwh` to build the IP driver map without touching
the running design. The instance names (`fft_dma`, `fft_wrapper_0`,
`dma_codec_mux_wrapper_0`, `sampler_codec_controller`) come from the block
design in `bd_project.tcl`.

In [ ]:
ol = Overlay(BITSTREAM, download=False)   # attach only; don't reprogram the PL

dma         = ol.fft_dma                    # axi_dma (we only use S2MM here)
fft_wrapper = ol.fft_wrapper_0              # HLS core (AXI-Lite: window coeffs)
mux         = ol.dma_codec_mux_wrapper_0    # selects DMA vs CODEC as FFT input
codec_ip    = ol.sampler.codec_controller   # SSM2603 controller

print(ol.ip_dict.keys())

## 2. Configure the codec

The codec has to be initialised before it produces audio on its output stream.
`CodecController.init()` runs the full SSM2603 bring-up (reset, power up, DSP
master mode, ~44.1 kHz, unmute, activate) — see `codec_config_demo.ipynb` for
the details. After this, the ADC/line-in is streaming into the PL.

In [ ]:
codec = CodecController(codec_ip)
codec.init()

## 3. Switch the mux to the codec

`dma_codec_mux_wrapper_0` selects which AXI-stream drives the FFT input:
`0` = DMA (`MM2S`), `1` = live codec. We select the codec here.

In [ ]:
mux.write(DMA_MUX_SEL_OFFSET, MUX_SEL_CODEC)   # 1 = CODEC
print(f'DMA_MUX_SEL = {mux.read(DMA_MUX_SEL_OFFSET)}  (1 = codec, 0 = dma)')

## 4. Program the window and arm the FFT

Same window coefficients as the DMA demo (Hann; the core mirrors the `N/2`
values). We arm `fft_wrapper` with `0x81` = `ap_start | auto_restart`, so it
keeps transforming successive frames of the continuous codec stream.

In [ ]:
# Hann window over the full N points; the core mirrors it, so it only needs N/2.
hann = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N) / (N - 1))
coeffs = hann[: N // 2]
max_q = (1 << WIN_FRAC) - 1
coeffs_q = np.clip(np.round(coeffs * (1 << WIN_FRAC)), 0, max_q).astype(np.uint32)
for i, c in enumerate(coeffs_q):
    fft_wrapper.write(WIN_COEFF_OFFSET + 4 * i, int(c))
print('wrote', len(coeffs_q), 'window coefficients')

# Arm with auto-restart so the FFT free-runs on the codec stream.
fft_wrapper.write(AP_CTRL_OFFSET, 0x81)
print(f'AP_CTRL = 0x{fft_wrapper.read(AP_CTRL_OFFSET):02x}')

## 5. Capture one FFT frame

With the codec free-running, we only arm the receive channel and grab the next
frame: 256 beats × 64 bits = 2048 bytes, allocated as `int32` of length `2*N`
so the interleaved `[real, imag]` pairs drop straight out.

In [ ]:
out_buf = allocate(shape=(2 * N,), dtype=np.int32)   # [re0, im0, re1, im1, ...]

dma.recvchannel.transfer(out_buf)
dma.recvchannel.wait()

raw = np.array(out_buf).reshape(N, 2).astype(np.float64) / OUT_SCALE
spectrum = raw[:, 0] + 1j * raw[:, 1]
print('captured one frame')

## 6. Plot the spectrum

One-sided magnitude; bin `k` maps to `k * FS / N` Hz. Whatever is on the codec
line-in / mic shows up here — feed it a tone and the peak lands at that
frequency.

In [ ]:
half  = N // 2
freqs = np.arange(half) * FS / N
mag   = np.abs(spectrum[:half])
mag_db = 20 * np.log10(mag + 1e-12)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(freqs, mag)
ax1.set_ylabel('|X(f)|'); ax1.set_title('FFT magnitude (from CODEC)'); ax1.grid(True)
ax2.plot(freqs, mag_db)
ax2.set_ylabel('dB'); ax2.set_xlabel('frequency [Hz]'); ax2.grid(True)
plt.tight_layout(); plt.show()

peak = freqs[np.argmax(mag)]
print(f'peak bin at {peak:.1f} Hz')

## 7. Average a few frames (optional)

Live audio is noisy; averaging the magnitude over several captured frames gives
a cleaner picture. Each `transfer/wait` grabs the next frame the FFT emits.

In [ ]:
K = 16
acc = np.zeros(half)
for _ in range(K):
    dma.recvchannel.transfer(out_buf)
    dma.recvchannel.wait()
    r = np.array(out_buf).reshape(N, 2).astype(np.float64) / OUT_SCALE
    acc += np.abs(r[:half, 0] + 1j * r[:half, 1])
avg = acc / K

plt.figure(figsize=(9, 3))
plt.plot(freqs, 20 * np.log10(avg + 1e-12))
plt.title(f'averaged spectrum ({K} frames)')
plt.xlabel('frequency [Hz]'); plt.ylabel('dB'); plt.grid(True)
plt.tight_layout(); plt.show()

print(f'peak bin at {freqs[np.argmax(avg)]:.1f} Hz')

## 8. Clean up

Free the DMA buffer and put the mux back to the DMA input so `fft_dma_demo.ipynb`
works again without reprogramming the PL.

In [ ]:
out_buf.freebuffer()
mux.write(DMA_MUX_SEL_OFFSET, MUX_SEL_DMA)   # restore DMA as FFT input
print(f'DMA_MUX_SEL = {mux.read(DMA_MUX_SEL_OFFSET)}  (back to dma)')